# Mapping Metabolic Plasticity in HGSOC
## Single-Cell Metabolic Landscape Pipeline
**Author:** Thomas Laronche (Project 20)
**Client:** Dr. Géraldine Gentric (MetaboCare, IGR)

This notebook contains the optimized, chunked pipeline to calculate 85 canonical metabolic pathway activities on a large-scale (390k cells) AnnData object without memory crashes.

In [1]:
import anndata as ad
import pandas as pd
import numpy as np
import gseapy as gp
from tqdm import tqdm

ModuleNotFoundError: No module named 'anndata'

### Step 1: Download MSigDB Canonical Pathways (C2:CP)

In [ ]:
msigdb_libs = ['KEGG_2021_Human', 'Reactome_2022']
pathways_raw = {}
for lib in msigdb_libs:
    try:
        gene_sets = gp.get_library(name=lib, organism='Human')
        pathways_raw.update(gene_sets)
    except Exception as e:
        pass

keywords = ["METABOLISM", "GLYCOLYSIS", "TCA", "CYCLE", "OXIDATIVE", "PHOSPHORYLATION", "LIPID", "AMINO", "FATTY", "SUGAR"]
pathways = {}
for pw_name, genes in pathways_raw.items():
    if any(kw in pw_name.upper() for kw in keywords):
        pathways[pw_name.upper()] = [g.strip().upper() for g in genes]

sorted_pw = sorted(pathways.items(), key=lambda item: len(item[1]), reverse=True)
pathways = dict(sorted_pw[:85])
print(f"-> {len(pathways)} canonical metabolic pathways loaded.")

### Step 2: Chunked Scoring (RAM Optimization)

In [ ]:
adata = ad.read_h5ad("data/raw/dataset_newyork.rds", backed='r')
n_obs = adata.n_obs

gene_symbols = [str(g).strip().upper() for g in adata.var['feature_name']]
gene_to_idx = {gene: idx for idx, gene in enumerate(gene_symbols)}

pw_indices = {}
for pw, genes in pathways.items():
    idxs = [gene_to_idx[g] for g in genes if g in gene_to_idx]
    if len(idxs) >= 3:
        pw_indices[pw] = idxs

valid_pws = list(pw_indices.keys())

scores_dict = {pw: np.zeros(n_obs, dtype=np.float32) for pw in valid_pws}
chunk_size = 10000

for start in tqdm(range(0, n_obs, chunk_size), desc="Scoring by chunks"):
    end = min(start + chunk_size, n_obs)
    chunk_X = adata.X[start:end]
    if hasattr(chunk_X, "toarray"):
        chunk_X = chunk_X.toarray()
    for pw in valid_pws:
        scores_dict[pw][start:end] = np.mean(chunk_X[:, pw_indices[pw]], axis=1)

### Step 3: Injection into Light AnnData

In [ ]:
adata_light = ad.read_h5ad("results/cellxgene/HGSOC_Metabolic_Scored_390k_Light.h5ad")

cols_to_drop = [c for c in adata_light.obs.columns if c.startswith("Meta_")]
adata_light.obs.drop(columns=cols_to_drop, inplace=True, errors='ignore')

for pw_name in valid_pws:
    safe_pw_name = pw_name.replace(" ", "_").replace("-", "_")
    adata_light.obs[f"Meta_{safe_pw_name}"] = scores_dict[pw_name]

adata_light.write_h5ad("results/cellxgene/HGSOC_Metabolic_Scored_390k_Light.h5ad")
print("✅ Success: Metadata injected.")